# Sydney hourly weather · Exploratory data analysis

**36120 Advanced Machine Learning Application** · Assessment Task 2 · Weather Intelligence (Open-Meteo, Sydney)

*Data notebook 2 of 3 · EDA*

Characterises the hourly observations **before** any target or feature is built. This notebook is descriptive only: it does not change the raw file.

### Student details

| | |
|:--|:--|
| **Student name** | Juan Vargas T. |
| **Student ID** | 25945320 |
| **Subject** | 36120 Advanced Machine Learning Application |
| **Assessment** | AT2 · Weather Intelligence |
| **GitHub** | [Sebas102507](https://github.com/Sebas102507) |

## Overview

### Questions addressed

1. What period and variables are available, and are values missing or duplicated?
2. How do the variables distribute, relate to each other and change through the seasons?
3. Which tail events, zeros or data limitations matter before building daily targets?

### Data dictionary

| Column | Unit | Description | Used by |
|:--|:--:|:--|:--|
| `time` | local time | Start of the hour, Sydney time (stored with its UTC offset) | Index |
| `temperature_2m` | °C | Air temperature at 2 m | CCI, WHC |
| `relative_humidity_2m` | % | Relative humidity at 2 m | CCI |
| `wind_speed_10m` | km/h | Wind speed at 10 m | CCI |
| `cloud_cover` | % | Share of sky covered by cloud | CCI, WHC |
| `precipitation` | mm | Total precipitation in the hour | CCI, WHC |
| `wind_gusts_10m` | km/h | Maximum wind gust at 10 m | WHC |
| `snowfall` | cm | Total snowfall in the hour | WHC |

> The targets are built from **daily** aggregates, so the hourly plots here describe the raw data;
> daily-scale behaviour is examined in section 5 and in the preprocessing notebook.

---
## 0. Setup

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from nucleus_ml import weather as wx

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", "{:.3f}".format)

PATHS = wx.paths()
RAW_CSV = PATHS["raw"] / "sydney_hourly.csv"
FIGURES = "eda"                       # figures are saved to reports/figures/eda/

WEATHER_COLUMNS = list(wx.HOURLY_VARIABLES)
DISPLAY_NAMES = {
    "temperature_2m": "Temperature (°C)",
    "relative_humidity_2m": "Relative humidity (%)",
    "wind_speed_10m": "Wind speed (km/h)",
    "cloud_cover": "Cloud cover (%)",
    "precipitation": "Precipitation (mm/h)",
    "wind_gusts_10m": "Wind gusts (km/h)",
    "snowfall": "Snowfall (cm/h)",
}

---
## 1. Load and validate

The raw file is read with the generic `wx.load_file`, and timestamps are converted to Sydney local
time with `wx.to_local_time`.

In [ ]:
hourly = wx.load_file(RAW_CSV)
hourly["time"] = wx.to_local_time(hourly["time"])
hourly = hourly.sort_values("time").reset_index(drop=True)

missing = sorted(set(["time", *WEATHER_COLUMNS]) - set(hourly.columns))
if missing:
    raise ValueError(f"Source is missing expected columns: {missing}")

print(f"Rows      : {len(hourly):,}")
print(f"Window    : {hourly['time'].min():%Y-%m-%d %H:%M} to {hourly['time'].max():%Y-%m-%d %H:%M}")
print(f"Duplicates: {hourly['time'].duplicated().sum():,} timestamps")
hourly.head()

---
## 2. Completeness and descriptive statistics

Missing values are reported per column. High quantiles are included because they reveal the
extreme events that a mean and standard deviation hide.

In [ ]:
quality = pd.DataFrame({
    "missing": hourly[WEATHER_COLUMNS].isna().sum(),
    "missing_pct": hourly[WEATHER_COLUMNS].isna().mean() * 100,
    "zero_pct": hourly[WEATHER_COLUMNS].eq(0).mean() * 100,
})
display(quality)

summary = hourly[WEATHER_COLUMNS].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T
summary[["min", "1%", "mean", "50%", "95%", "99%", "max"]]

---
## 3. Time coverage and daylight saving

Sydney's local calendar has a 23-hour day in October and a 25-hour day in April. These are
expected; any other day with a different hour count, or any missing hour, would be a genuine gap
that must be handled before daily aggregation.

In [ ]:
expected = pd.date_range(hourly["time"].min(), hourly["time"].max(), freq="h")
missing_hours = expected.difference(pd.DatetimeIndex(hourly["time"]))
hours_per_day = hourly.groupby(hourly["time"].dt.normalize()).size()

print(f"Missing hours: {len(missing_hours):,}")
display(hours_per_day[hours_per_day != 24].rename("hours").to_frame().head(10))

fig, ax = plt.subplots(figsize=(12, 3))
hours_per_day.plot(ax=ax, lw=0.5)
ax.axhline(24, color="black", ls="--", lw=1)
ax.set(title="Hourly observations per local day", xlabel="Date", ylabel="Hours")
fig.tight_layout()
wx.save_figure(fig, "eda-hours-per-day", FIGURES)
plt.show()

---
## 4. Long-run behaviour, distributions and seasonality

Monthly averages show seasonality and any long-term trend. Histograms use the raw hourly scale.
Snowfall is left out of the plots because it is always zero in Sydney.

The correlation matrix is **descriptive only** (same-hour Pearson correlation). It says nothing
about predicting future days and is not used to choose features.

In [ ]:
PLOT_COLUMNS = [c for c in WEATHER_COLUMNS if c != "snowfall"]
by_time = hourly.set_index("time")[PLOT_COLUMNS]

fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
for ax, column in zip(axes.ravel(), PLOT_COLUMNS):
    by_time[column].resample("MS").mean().plot(ax=ax, lw=1)
    ax.set(title=DISPLAY_NAMES[column], xlabel="", ylabel="Monthly mean")
fig.suptitle("Monthly average conditions", y=1.01)
fig.tight_layout()
wx.save_figure(fig, "eda-monthly-conditions", FIGURES)
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, column in zip(axes.ravel(), PLOT_COLUMNS):
    ax.hist(hourly[column].dropna(), bins=40, color="tab:blue", edgecolor="white")
    ax.set(title=DISPLAY_NAMES[column], xlabel="Hourly value", ylabel="Hours")
fig.suptitle("Hourly distributions", y=1.02)
fig.tight_layout()
wx.save_figure(fig, "eda-hourly-distributions", FIGURES)
plt.show()

In [ ]:
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
annual_cycle = hourly.groupby(hourly["time"].dt.month)[PLOT_COLUMNS].mean()

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, column in zip(axes.ravel(), ["temperature_2m", "relative_humidity_2m",
                                     "wind_speed_10m", "cloud_cover"]):
    annual_cycle[column].plot(ax=ax, marker="o")
    ax.set(title=DISPLAY_NAMES[column], xlabel="Month", ylabel="Mean")
    ax.set_xticks(range(1, 13), MONTHS)
fig.suptitle("Mean annual cycle", y=1.02)
fig.tight_layout()
wx.save_figure(fig, "eda-seasonal-cycle", FIGURES)
plt.show()

correlations = hourly[PLOT_COLUMNS].corr()
fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(correlations, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(PLOT_COLUMNS)), PLOT_COLUMNS, rotation=45, ha="right")
ax.set_yticks(range(len(PLOT_COLUMNS)), PLOT_COLUMNS)
for i in range(len(PLOT_COLUMNS)):
    for j in range(len(PLOT_COLUMNS)):
        ax.text(j, i, f"{correlations.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(image, ax=ax, label="Pearson correlation")
ax.set_title("Same-hour correlations")
fig.tight_layout()
wx.save_figure(fig, "eda-correlations", FIGURES)
plt.show()

---
## 5. Rare events at the daily scale

The formula constants refer to **daily** quantities (20 mm for `RainScore`, 30 mm for `RainHazard`,
100 km/h for gusts). Daily rain is therefore summed and gusts take the daily maximum, the same
rules the preprocessing pipeline uses. The reference lines show where each score saturates.

In [ ]:
daily = hourly.set_index("time").resample("D").agg(
    {"precipitation": "sum", "wind_gusts_10m": "max", "snowfall": "sum"})

events = pd.Series({
    "days with precipitation >= 20 mm": int((daily["precipitation"] >= 20).sum()),
    "days with precipitation >= 30 mm": int((daily["precipitation"] >= 30).sum()),
    "days with gusts >= 100 km/h": int((daily["wind_gusts_10m"] >= 100).sum()),
    "distinct snowfall values": int(hourly["snowfall"].nunique()),
}, name="count")
display(events.to_frame())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(daily["precipitation"], bins=50, color="tab:blue", edgecolor="white")
axes[0].axvline(20, color="tab:orange", ls="--", label="20 mm (RainScore = 0)")
axes[0].axvline(30, color="tab:red", ls="--", label="30 mm (RainHazard = 1)")
axes[0].set(title="Daily precipitation", xlabel="mm/day", ylabel="Days")
axes[1].hist(daily["wind_gusts_10m"], bins=50, color="tab:blue", edgecolor="white")
axes[1].axvline(100, color="tab:red", ls="--", label="100 km/h (WindHazard = 1)")
axes[1].set(title="Daily maximum wind gust", xlabel="km/h", ylabel="Days")
for ax in axes:
    ax.legend()
fig.tight_layout()
wx.save_figure(fig, "eda-daily-tail-events", FIGURES)
plt.show()

daily[["precipitation", "wind_gusts_10m"]].quantile([0.5, 0.9, 0.95, 0.99, 1.0]).T

---
## 6. Conclusions

Replace each point with the observed values when writing the report:

- **Completeness.** Confirm every required column is complete and that the only irregular days are
  daylight-saving transitions (sections 2–3).
- **Zero-inflated variables.** Precipitation is mostly zero with a long right tail; snowfall is
  always zero, so its WHC hazard term is constant.
- **Rare hazards.** Few days reach the 30 mm rain or 100 km/h gust caps, so the upper hazard
  classes will be rare — the key challenge for the classification task.
- **Time order.** Every later lag or rolling feature may use only day *t* and earlier; targets are
  shifted to their future horizons in the preprocessing notebook.